## Exercise 1

**Dataset Used:** Custom/Synthetic Array Data (numpy)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 4: Aktivierungsfunktionen
# Niveau: Fortgeschrittene
# Aufgabe 1 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

# Demonstrate and solve the "dying ReLU" problem
import numpy as np
import matplotlib

import matplotlib.pyplot as plt
import tensorflow as tf

tf.random.set_seed(42); np.random.seed(42)

# Synthetische Daten
X = np.random.randn(200, 10).astype(np.float32)
y = (X[:,0] + X[:,1] > 0).astype(np.float32)

def erstelle_modell_relu(aktivierung_name):
    """Erstellt ein Modell mit standardmäßiger ReLU oder ELU"""
    m = tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation=aktivierung_name, input_shape=(10,),
                              kernel_initializer='he_normal'),
        tf.keras.layers.Dense(64, activation=aktivierung_name),
        tf.keras.layers.Dense(1, activation='sigmoid'),
    ])
    m.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=0.1),
              loss='binary_crossentropy', metrics=['accuracy'])
    return m

def erstelle_modell_leaky():
    """Erstellt ein Modell mit Leaky ReLU"""
    m = tf.keras.Sequential([
        tf.keras.layers.Dense(64, input_shape=(10,), kernel_initializer='he_normal'),
        tf.keras.layers.LeakyReLU(negative_slope=0.01),
        tf.keras.layers.Dense(64),
        tf.keras.layers.LeakyReLU(negative_slope=0.01),
        tf.keras.layers.Dense(1, activation='sigmoid'),
    ])
    m.compile(optimizer=tf.keras.optimizers.SGD(0.1), loss='binary_crossentropy', metrics=['accuracy'])
    return m

vergleich = {}

# ReLU (kann tote Neuronen haben bei hoher Lernrate)
m_relu = erstelle_modell_relu('relu')
h_relu = m_relu.fit(X, y, epochs=50, verbose=0)
vergleich['relu'] = h_relu

# Leaky ReLU
m_leaky = erstelle_modell_leaky()
h_leaky = m_leaky.fit(X, y, epochs=50, verbose=0)
vergleich['leaky_relu'] = h_leaky

# ELU
m_elu = erstelle_modell_relu('elu')
h_elu = m_elu.fit(X, y, epochs=50, verbose=0)
vergleich['elu'] = h_elu

fig, ax = plt.subplots(figsize=(10, 5))
for name, h in vergleich.items():
    ax.plot(h.history['loss'], label=name, linewidth=2)
ax.set_title('Sterbendes ReLU: Verlust nach Aktivierungsfunktion')
ax.set_xlabel('Epoche'); ax.set_ylabel('Verlust')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('sterbendes_relu.png', dpi=100)
plt.show()
print("Leaky ReLU / ELU verhindern das 'dying ReLU' Problem!")
print("Gespeichert: sterbendes_relu.png")


## Exercise 2

**Dataset Used:** Custom/Synthetic Array Data (numpy)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 4: Aktivierungsfunktionen
# Niveau: Fortgeschrittene
# Aufgabe 2 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

# Deep dive into Softmax: temperature scaling and calibration
import numpy as np
import matplotlib

import matplotlib.pyplot as plt

def softmax(logits, temperatur=1.0):
    """Softmax mit Temperaturparameter für Kalibrierung"""
    logits_skaliert = logits / temperatur
    exp_l = np.exp(logits_skaliert - np.max(logits_skaliert))
    return exp_l / exp_l.sum()

logits = np.array([2.0, 1.0, 0.5, -0.5, -1.0])
temperaturen = [0.1, 0.5, 1.0, 2.0, 5.0]
klassen = [f'K{i+1}' for i in range(len(logits))]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Wahrscheinlichkeiten für verschiedene Temperaturen
for T in temperaturen:
    prob = softmax(logits, T)
    axes[0].plot(klassen, prob, 'o-', linewidth=2, markersize=8, label=f'T={T}')

axes[0].set_title('Softmax mit verschiedenen Temperaturen')
axes[0].set_xlabel('Klasse'); axes[0].set_ylabel('Wahrscheinlichkeit')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# Maximale Wahrscheinlichkeit und Entropie vs. Temperatur
t_vals = np.linspace(0.1, 10, 100)
max_probs = [softmax(logits, T).max() for T in t_vals]
entropien = [-sum(p*np.log(p+1e-15) for p in softmax(logits, T)) for T in t_vals]

axes[1].plot(t_vals, max_probs, 'b-', linewidth=2, label='Max. Wahrscheinlichkeit')
ax2 = axes[1].twinx()
ax2.plot(t_vals, entropien, 'r--', linewidth=2, label='Entropie')
axes[1].set_xlabel('Temperatur T')
axes[1].set_ylabel('Max. Wahrscheinlichkeit', color='blue')
ax2.set_ylabel('Entropie', color='red')
axes[1].set_title('Effekt der Temperatur auf Konfidenz')
axes[1].grid(True, alpha=0.3)

# Legenden zusammenführen
lines1, labels1 = axes[1].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
axes[1].legend(lines1 + lines2, labels1 + labels2, loc='center right')

plt.tight_layout()
plt.savefig('softmax_temperatur.png', dpi=100)
plt.show()
print(f"T=0.1 (sicher): {softmax(logits, 0.1).round(3)}")
print(f"T=1.0 (normal): {softmax(logits, 1.0).round(3)}")
print(f"T=5.0 (weich):  {softmax(logits, 5.0).round(3)}")


## Exercise 3

**Dataset Used:** Custom/Synthetic Array Data (numpy)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 4: Aktivierungsfunktionen
# Niveau: Fortgeschrittene
# Aufgabe 3 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

# Analyze gradient flow with different activation functions
import numpy as np
import matplotlib

import matplotlib.pyplot as plt
import tensorflow as tf

tf.random.set_seed(42); np.random.seed(42)

X = np.random.randn(100, 2).astype(np.float32)
y = (X[:,0] + X[:,1] > 0).astype(np.float32)

def erstelle_tiefes_netz(aktivierung, tiefe=10):
    """Erstellt tiefes Netz mit gewählter Aktivierungsfunktion"""
    schichten = [tf.keras.layers.Dense(32, activation=aktivierung, input_shape=(2,),
                                        kernel_initializer='glorot_uniform')]
    for _ in range(tiefe-1):
        schichten.append(tf.keras.layers.Dense(32, activation=aktivierung,
                                               kernel_initializer='glorot_uniform'))
    schichten.append(tf.keras.layers.Dense(1, activation='sigmoid'))
    m = tf.keras.Sequential(schichten)
    m.compile(optimizer='adam', loss='binary_crossentropy')
    return m

def berechne_gradienten(modell, X, y):
    """Berechnet mittlere absolute Gradienten pro Schicht"""
    with tf.GradientTape() as tape:
        pred = modell(X, training=True)
        verlust = tf.keras.losses.binary_crossentropy(y, pred[:,0])
    gradienten = tape.gradient(verlust, modell.trainable_variables)
    return [float(tf.reduce_mean(tf.abs(g)).numpy()) for g in gradienten if len(g.shape) == 2]

aktivierungen = ['sigmoid', 'tanh', 'relu', 'elu']
fig, ax = plt.subplots(figsize=(12, 6))

for akt in aktivierungen:
    modell = erstelle_tiefes_netz(akt)
    grads = berechne_gradienten(modell, X, y)
    ax.plot(range(len(grads)), grads, 'o-', linewidth=2, markersize=8, label=akt)

ax.set_xlabel('Schichtnummer (von Ausgabe zu Eingabe)')
ax.set_ylabel('Mittlere absolute Gradientenstärke')
ax.set_title('Gradientenfluss durch 10-schichtiges Netz\n(vor dem Training)')
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_yscale('log')
plt.tight_layout()
plt.savefig('gradientenfluss.png', dpi=100)
plt.show()
print("Gradientenfluss gespeichert: gradientenfluss.png")
print("Sigmoid zeigt stärksten Gradientschwund – ReLU/ELU sind deutlich besser!")
